# Intelligent Final Year Project Recommendation System

This notebook implements the recommender system for the final-year project titled:
**Design and Implementation of an Intelligent Final Year Project Recommendation System Using Student Academic Performance**.

The notebook loads the student and project datasets, engineers student and project profile features, trains an XGBoost binary recommender, evaluates the model, and saves the inference artifacts.

## 1. Setup and Imports

In [1]:
import json
import os
import pickle
import re
from datetime import datetime

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

ROOT_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(ROOT_DIR, 'dataset')
MODELS_DIR = os.path.join(ROOT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

STUDENTS_PATH = os.path.join(DATA_DIR, 'students.csv')
PROJECTS_PATH = os.path.join(DATA_DIR, 'projects.csv')
HISTORY_PATH = os.path.join(DATA_DIR, 'history.csv')

## 2. Data Loading and Inspection

In [2]:
students = pd.read_csv(STUDENTS_PATH)
projects = pd.read_csv(PROJECTS_PATH)
history = pd.read_csv(HISTORY_PATH)

print('Students', students.shape)
print('Projects', projects.shape)
print('History', history.shape)

students.head()

Students (500, 12)
Projects (1628, 10)
History (995, 7)


,student_id,department,year_level,gpa,avg_cs_grade,programming_languages,frameworks_tools,interests,preferred_difficulty,past_projects,num_past_projects,past_project_grades
0,STU-0001,Computer Science,3,3.42,3.58,"JavaScript, C++, Java, Scala, PHP","React, Hadoop, Linux, Splunk, Angular, TensorFlow","deep learning, machine learning",Medium,Active Site Identification Algorithm; Web Scra...,2,2.43; 3.68
1,STU-0002,Software Engineering,4,2.60,2.11,"C#, R, C++","PostgreSQL, Kubernetes, Django","API development, devops, UI/UX design",Easy,Currency Converter for Exchange Rate Conversion,1,2.46
2,STU-0003,Information Systems,4,3.68,3.47,"Java, C#, MATLAB","Linux, Vue.js, Scikit-learn, MySQL","healthcare systems, retail systems, customer r...",Hard,Invoice Generator System for Billing and Accou...,3,4.13; 4.22; 3.67
3,STU-0004,Computer Science,4,3.87,3.92,"Go, TypeScript, Ruby","AWS, Django, Keras","quantum computing, generative AI, reinforcemen...",Hard,Seasonal Time Series Decomposition Framework f...,3,4.1; 3.55; 4.09
4,STU-0005,Information Systems,3,2.48,2.56,"JavaScript, PHP","Kubernetes, Splunk, MongoDB, Azure, Node.js","customer relationship management, data analyti...",Medium,Customer Churn Prediction System for Retention...,1,2.0


## 3. Feature Engineering Helpers

In [3]:
TECHNICAL_SKILLS = {
    'python': ['python'],
    'java': ['java'],
    'cpp': ['c++', 'cpp'],
    'javascript': ['javascript', 'js', 'node'],
    'sql': ['sql', 'postgresql', 'mysql', 'sqlite', 'nosql', 'mongodb'],
    'machine_learning': ['machine learning', 'ml', 'scikit-learn', 'tensorflow', 'pytorch', 'keras'],
    'deep_learning': ['deep learning', 'neural network', 'cnn', 'rnn', 'lstm', 'transformer', 'bert'],
    'networking': ['networking', 'network', 'tcp', 'udp', 'routing', 'switching'],
    'cybersecurity': ['cybersecurity', 'cyber security', 'security', 'penetration', 'intrusion', 'forensics', 'incident response'],
    'cloud_computing': ['cloud', 'aws', 'azure', 'gcp', 'google cloud', 'cloud computing'],
    'mobile_development': ['mobile', 'android', 'ios', 'flutter', 'react native', 'kotlin', 'swift'],
    'database_management': ['database', 'db', 'sql', 'nosql', 'mongodb', 'postgresql', 'mysql', 'oracle'],
    'software_engineering': ['software engineering', 'software development', 'architecture', 'design pattern', 'oop'],
}

DOMAIN_INTERESTS = {
    'artificial_intelligence': ['artificial intelligence', 'ai', 'machine learning', 'deep learning', 'neural', 'computer vision', 'nlp'],
    'data_science': ['data science', 'data analytics', 'analytics', 'data mining', 'visualization', 'big data'],
    'cybersecurity': ['cybersecurity', 'security', 'crypto', 'penetration', 'incident response', 'forensics'],
    'web_development': ['web development', 'frontend', 'backend', 'full stack', 'react', 'angular', 'vue', 'html', 'css', 'javascript'],
    'mobile_development': ['mobile development', 'android', 'ios', 'flutter', 'react native', 'kotlin', 'swift'],
    'nlp': ['nlp', 'natural language processing', 'language model', 'text classification', 'sentiment'],
    'computer_vision': ['computer vision', 'image recognition', 'object detection', 'segmentation', 'opencv'],
    'iot': ['iot', 'internet of things', 'embedded systems', 'raspberry', 'arduino'],
    'blockchain': ['blockchain', 'smart contract', 'ethereum', 'bitcoin', 'crypto'],
    'embedded_systems': ['embedded systems', 'microcontroller', 'firmware', 'iot', 'raspberry', 'arduino'],
    'software_engineering': ['software engineering', 'software development', 'architecture', 'testing', 'devops'],
}

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def keyword_flags(text, keyword_map):
    text = clean_text(text)
    flags = {}
    for name, patterns in keyword_map.items():
        flags[name] = int(any(pattern in text for pattern in patterns))
    return flags

def build_student_records(students):
    students = students.copy()
    students['department_clean'] = students['department'].astype(str)
    students['pref_diff_clean'] = students['preferred_difficulty'].astype(str)
    students['gpa_norm'] = MinMaxScaler().fit_transform(students[['gpa']])
    students['avg_cs_grade_norm'] = MinMaxScaler().fit_transform(students[['avg_cs_grade']])
    students['interest_text'] = students['interests'].fillna('').astype(str)
    students['skills_text'] = (students['programming_languages'].fillna('') + ' ' + students['frameworks_tools'].fillna('')).astype(str)
    students['skill_flags'] = students['skills_text'].apply(lambda text: keyword_flags(text, TECHNICAL_SKILLS))
    students['domain_flags'] = students['interest_text'].apply(lambda text: keyword_flags(text, DOMAIN_INTERESTS))
    students['past_projects_text'] = students['past_projects'].fillna('').astype(str)
    return students

def build_project_records(projects):
    projects = projects.copy()
    projects['department_clean'] = projects['department'].astype(str)
    projects['difficulty_clean'] = projects['difficulty'].astype(str)
    projects['category_clean'] = projects['category'].astype(str)
    projects['avg_grade_norm'] = MinMaxScaler().fit_transform(projects[['avg_grade_given']])
    projects['times_selected_norm'] = MinMaxScaler().fit_transform(projects[['times_selected']])
    projects['project_text'] = (
        projects['title'].fillna('') + ' ' +
        projects['description'].fillna('') + ' ' +
        projects['required_skills'].fillna('') + ' ' +
        projects['tech_stack'].fillna('') + ' ' +
        projects['category'].fillna('')
    ).astype(str)
    projects['skill_text'] = (projects['required_skills'].fillna('') + ' ' + projects['tech_stack'].fillna('')).astype(str)
    projects['skill_flags'] = projects['skill_text'].apply(lambda text: keyword_flags(text, TECHNICAL_SKILLS))
    projects['domain_flags'] = projects['project_text'].apply(lambda text: keyword_flags(text, DOMAIN_INTERESTS))
    return projects

def build_similarity_features(student_row, project_row, student_text_vector, project_text_vector):
    student_skills = np.array(list(student_row['skill_flags'].values()), dtype=int)
    project_skills = np.array(list(project_row['skill_flags'].values()), dtype=int)
    skill_overlap = int(np.logical_and(student_skills, project_skills).sum())
    skill_union = int(np.logical_or(student_skills, project_skills).sum())
    skill_overlap_ratio = skill_overlap / max(skill_union, 1)

    student_domains = np.array(list(student_row['domain_flags'].values()), dtype=int)
    project_domains = np.array(list(project_row['domain_flags'].values()), dtype=int)
    domain_overlap = int(np.logical_and(student_domains, project_domains).sum())
    domain_union = int(np.logical_or(student_domains, project_domains).sum())
    domain_overlap_ratio = domain_overlap / max(domain_union, 1)

    dept_match = float(student_row['department_clean'] == project_row['department_clean'])
    diff_match = 1.0 - abs(student_row['preferred_diff_encoded'] - project_row['difficulty_encoded']) / 2.0
    category_match = float(project_row['category_clean'].lower() in student_row['interest_text'].lower())

    student_vec = student_text_vector.toarray()[0]
    project_vec = project_text_vector.toarray()[0]
    interest_similarity = float(
        np.dot(student_vec, project_vec) /
        (np.linalg.norm(student_vec) * np.linalg.norm(project_vec) + 1e-9)
    )

    return {
        'skill_overlap': skill_overlap,
        'skill_overlap_ratio': skill_overlap_ratio,
        'domain_overlap': domain_overlap,
        'domain_overlap_ratio': domain_overlap_ratio,
        'dept_match': dept_match,
        'diff_match': diff_match,
        'category_match': category_match,
        'interest_similarity': interest_similarity,
    }

def build_feature_vector(student_row, project_row, similarity_features):
    features = [
        student_row['department_encoded'],
        student_row['year_level'],
        student_row['gpa_norm'],
        student_row['avg_cs_grade_norm'],
        student_row['preferred_diff_encoded'],
        student_row['num_past_projects'],
        project_row['department_encoded'],
        project_row['difficulty_encoded'],
        project_row['category_encoded'],
        project_row['avg_grade_norm'],
        project_row['times_selected_norm'],
        similarity_features['skill_overlap'],
        similarity_features['skill_overlap_ratio'],
        similarity_features['domain_overlap'],
        similarity_features['domain_overlap_ratio'],
        similarity_features['dept_match'],
        similarity_features['diff_match'],
        similarity_features['category_match'],
        similarity_features['interest_similarity'],
    ]
    features.extend(list(student_row['skill_flags'].values()))
    features.extend(list(project_row['skill_flags'].values()))
    features.extend(list(student_row['domain_flags'].values()))
    features.extend(list(project_row['domain_flags'].values()))
    return np.array(features, dtype=float)

## 4. Prepare Student and Project Profiles

In [4]:
students = build_student_records(students)
projects = build_project_records(projects)

dept_encoder = LabelEncoder().fit(students['department_clean'])
diff_encoder = LabelEncoder().fit(['Easy', 'Medium', 'Hard'])
category_encoder = LabelEncoder().fit(projects['category_clean'])

students['department_encoded'] = dept_encoder.transform(students['department_clean'])
students['preferred_diff_encoded'] = diff_encoder.transform(students['pref_diff_clean'].fillna('Easy'))
projects['department_encoded'] = dept_encoder.transform(projects['department_clean'])
projects['difficulty_encoded'] = diff_encoder.transform(projects['difficulty_clean'].fillna('Easy'))
projects['category_encoded'] = category_encoder.transform(projects['category_clean'])

print('Finished encoding student and project categorical features.')
students.head()

Finished encoding student and project categorical features.


,student_id,department,year_level,gpa,avg_cs_grade,programming_languages,frameworks_tools,interests,preferred_difficulty,past_projects,...,pref_diff_clean,gpa_norm,avg_cs_grade_norm,interest_text,skills_text,skill_flags,domain_flags,past_projects_text,department_encoded,preferred_diff_encoded
0,STU-0001,Computer Science,3,3.42,3.58,"JavaScript, C++, Java, Scala, PHP","React, Hadoop, Linux, Splunk, Angular, TensorFlow","deep learning, machine learning",Medium,Active Site Identification Algorithm; Web Scra...,...,Medium,0.653005,0.6450,"deep learning, machine learning","JavaScript, C++, Java, Scala, PHP React, Hadoo...","{'python': 0, 'java': 1, 'cpp': 0, 'javascript...","{'artificial_intelligence': 1, 'data_science':...",Active Site Identification Algorithm; Web Scra...,0,2
1,STU-0002,Software Engineering,4,2.60,2.11,"C#, R, C++","PostgreSQL, Kubernetes, Django","API development, devops, UI/UX design",Easy,Currency Converter for Exchange Rate Conversion,...,Easy,0.428962,0.2775,"API development, devops, UI/UX design","C#, R, C++ PostgreSQL, Kubernetes, Django","{'python': 0, 'java': 0, 'cpp': 0, 'javascript...","{'artificial_intelligence': 0, 'data_science':...",Currency Converter for Exchange Rate Conversion,4,0
2,STU-0003,Information Systems,4,3.68,3.47,"Java, C#, MATLAB","Linux, Vue.js, Scikit-learn, MySQL","healthcare systems, retail systems, customer r...",Hard,Invoice Generator System for Billing and Accou...,...,Hard,0.724044,0.6175,"healthcare systems, retail systems, customer r...","Java, C#, MATLAB Linux, Vue.js, Scikit-learn, ...","{'python': 0, 'java': 1, 'cpp': 0, 'javascript...","{'artificial_intelligence': 1, 'data_science':...",Invoice Generator System for Billing and Accou...,2,1
3,STU-0004,Computer Science,4,3.87,3.92,"Go, TypeScript, Ruby","AWS, Django, Keras","quantum computing, generative AI, reinforcemen...",Hard,Seasonal Time Series Decomposition Framework f...,...,Hard,0.775956,0.7300,"quantum computing, generative AI, reinforcemen...","Go, TypeScript, Ruby AWS, Django, Keras","{'python': 0, 'java': 0, 'cpp': 0, 'javascript...","{'artificial_intelligence': 1, 'data_science':...",Seasonal Time Series Decomposition Framework f...,0,1
4,STU-0005,Information Systems,3,2.48,2.56,"JavaScript, PHP","Kubernetes, Splunk, MongoDB, Azure, Node.js","customer relationship management, data analyti...",Medium,Customer Churn Prediction System for Retention...,...,Medium,0.396175,0.3900,"customer relationship management, data analyti...","JavaScript, PHP Kubernetes, Splunk, MongoDB, A...","{'python': 0, 'java': 1, 'cpp': 0, 'javascript...","{'artificial_intelligence': 1, 'data_science':...",Customer Churn Prediction System for Retention...,2,2


## 5. Build Training Dataset

In [5]:
text_vectorizer = TfidfVectorizer(max_features=200, stop_words='english')
combined_text = pd.concat([students['interest_text'], projects['project_text']], ignore_index=True)
combined_matrix = text_vectorizer.fit_transform(combined_text)
interest_matrix = combined_matrix[:len(students)]
project_matrix = combined_matrix[len(students):]

student_idx = {sid: i for i, sid in enumerate(students['student_id'])}
project_idx = {pid: i for i, pid in enumerate(projects['project_id'])}

positive_history = history[
    (history['completion_status'] == 'completed') &
    (history['rating'] >= 4)
]
positive_history = positive_history[positive_history['student_id'].isin(student_idx) & positive_history['project_id'].isin(project_idx)]

print(f'Positive interactions: {len(positive_history)}')

random_negatives = []
rng = np.random.default_rng(42)
observed_pairs = set(zip(history['student_id'], history['project_id']))
for _, row in positive_history.iterrows():
    sid = row['student_id']
    for _ in range(3):
        candidate = rng.choice(projects['project_id'].values)
        if (sid, candidate) in observed_pairs:
            continue
        random_negatives.append({'student_id': sid, 'project_id': candidate, 'label': 0})

negative_history = history[~history.index.isin(positive_history.index)].copy()
negative_history = negative_history[negative_history['student_id'].isin(student_idx) & negative_history['project_id'].isin(project_idx)]
negative_history = negative_history.assign(label=0)[['student_id', 'project_id', 'label']]

training_rows = []
training_rows.extend(positive_history.assign(label=1)[['student_id', 'project_id', 'label']].to_dict(orient='records'))
training_rows.extend(random_negatives)
training_rows.extend(negative_history.to_dict(orient='records'))

dataframe = pd.DataFrame(training_rows).drop_duplicates(['student_id', 'project_id', 'label'])
dataframe = dataframe.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Total training rows: {len(dataframe)}')

feature_vectors = []
labels = []

for _, row in dataframe.iterrows():
    s = row['student_id']
    p = row['project_id']
    student_row = students.loc[students['student_id'] == s].iloc[0]
    project_row = projects.loc[projects['project_id'] == p].iloc[0]
    student_idx_val = students.index[students['student_id'] == s][0]
    project_idx_val = projects.index[projects['project_id'] == p][0]
    similarity = build_similarity_features(
        student_row,
        project_row,
        interest_matrix[student_idx_val],
        project_matrix[project_idx_val],
    )
    vector = build_feature_vector(student_row, project_row, similarity)
    feature_vectors.append(vector)
    labels.append(row['label'])

X = np.vstack(feature_vectors)
y = np.array(labels, dtype=int)
print(f'Feature matrix shape: {X.shape}')

Positive interactions: 211
Total training rows: 1626
Feature matrix shape: (1626, 67)


## 6. Train the Recommender

In [6]:
feature_names = [
    'student_dept', 'student_year', 'student_gpa', 'student_cs_grade',
    'student_pref_diff', 'student_num_past', 'project_dept', 'project_diff',
    'project_category', 'project_avg_grade', 'project_times_selected',
    'skill_overlap', 'skill_overlap_ratio', 'domain_overlap', 'domain_overlap_ratio',
    'dept_match', 'diff_match', 'category_match', 'interest_similarity',
]
feature_names.extend([f'student_skill_{k}' for k in TECHNICAL_SKILLS])
feature_names.extend([f'project_skill_{k}' for k in TECHNICAL_SKILLS])
feature_names.extend([f'student_domain_{k}' for k in DOMAIN_INTERESTS])
feature_names.extend([f'project_domain_{k}' for k in DOMAIN_INTERESTS])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train size:', X_train.shape, 'Test size:', X_test.shape)

model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    'roc_auc': roc_auc_score(y_test, y_proba),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
}
print(metrics)

Train size: (1300, 67) Test size: (326, 67)
{'roc_auc': 0.8548792756539235, 'precision': 0.43333333333333335, 'recall': 0.30952380952380953, 'f1': 0.3611111111111111}


## 7. Top-5 Recommendation Example

In [ ]:
def recommend_projects(student_id, top_n=5):
    student_row = students[students['student_id'] == student_id].iloc[0]
    student_index = students.index[students['student_id'] == student_id][0]
    project_scores = []
    for project_index, project_row in projects.iterrows():
        similarity = build_similarity_features(
            student_row,
            project_row,
            interest_matrix[student_index],
            project_matrix[project_index],
        )
        vector = build_feature_vector(student_row, project_row, similarity).reshape(1, -1)
        score = model.predict_proba(vector)[0, 1]
        project_scores.append((project_row['project_id'], project_row['title'], score, similarity))

    ranked = sorted(project_scores, key=lambda x: x[2], reverse=True)[:top_n]
    return ranked

example_student = students['student_id'].iloc[0]
top_projects = recommend_projects(example_student, top_n=5)
for i, (pid, title, score, sim) in enumerate(top_projects, 1):
    print(f'{i}. {title} ({pid}) - Match: {score:.3f}')
    print(f'   Skill overlap: {sim["skill_overlap_ratio"]:.2f}, Domain overlap: {sim["domain_overlap_ratio"]:.2f}')

1. Ant Colony Optimization for Combinatorial Problems (COS-ALG-1164) - Match: 0.901


NameError: name 'skill_overlap_ratio' is not defined

## 8. Save Model Artifacts

In [ ]:
with open(os.path.join(MODELS_DIR, 'recommender.pkl'), 'wb') as f:
    pickle.dump(model, f)

with open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump({
        'dept_encoder': dept_encoder,
        'diff_encoder': diff_encoder,
        'category_encoder': category_encoder,
    }, f)

with open(os.path.join(MODELS_DIR, 'encoder.pkl'), 'wb') as f:
    pickle.dump({
        'technical_skills': TECHNICAL_SKILLS,
        'domain_interests': DOMAIN_INTERESTS,
    }, f)

with open(os.path.join(MODELS_DIR, 'vectorizer.pkl'), 'wb') as f:
    pickle.dump({
        'text_vectorizer': text_vectorizer,
    }, f)

with open(os.path.join(MODELS_DIR, 'metadata.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'model_type': 'xgboost_classifier',
        'feature_count': X.shape[1],
        'student_count': int(students.shape[0]),
        'project_count': int(projects.shape[0]),
        'training_rows': int(X.shape[0]),
    }, f, indent=2)

print('Saved artifacts to models/')